# EXP21: 最大持仓时间精细扫描 (KC + ORB 独立)

Timeout:60 bars 是最大单项亏损源 (-$12,287)。
KC max_hold [8,10,12,15,18,20,24,30] + ORB max_hold_m15 [8,12,16,20,24] 独立扫描

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import config
import pandas as pd

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Ready')

In [ ]:
# Part 1: KC max_hold scan
results_kc = []
orig_mh = config.STRATEGIES['keltner']['max_hold_bars']
for mh in [8, 10, 12, 15, 18, 20, 24, 30, 40, 60]:
    config.STRATEGIES['keltner']['max_hold_bars'] = mh
    r = run_variant(data, f"KC_mh={mh}", **LIVE_KWARGS)
    r['kc_max_hold'] = mh
    # count timeout exits
    timeouts = sum(1 for t in r['_trades'] if 'timeout' in t.exit_reason.lower() and t.strategy == 'keltner')
    r['kc_timeouts'] = timeouts
    results_kc.append(r)
    print(f"  KC mh={mh:2d}: Sharpe={r['sharpe']:.2f}, N={r['n']}, PnL=${r['total_pnl']:.0f}, Timeouts={timeouts}")

config.STRATEGIES['keltner']['max_hold_bars'] = orig_mh
print_ranked(results_kc)

In [ ]:
# Part 2: ORB max_hold scan (M15 bars)
results_orb = []
for mh in [8, 12, 16, 20, 24, 32, 48]:
    r = run_variant(data, f"ORB_mh={mh}", **{**LIVE_KWARGS, "orb_max_hold_m15": mh})
    r['orb_max_hold'] = mh
    orb_trades = [t for t in r['_trades'] if t.strategy == 'orb']
    orb_pnl = sum(t.pnl for t in orb_trades)
    r['orb_pnl'] = orb_pnl
    r['orb_n'] = len(orb_trades)
    results_orb.append(r)
    print(f"  ORB mh={mh:2d}: Sharpe={r['sharpe']:.2f}, ORB PnL=${orb_pnl:.0f} ({len(orb_trades)} trades)")

print_ranked(results_orb)

In [ ]:
# Part 3: KC × ORB cross grid (top 3 each)
kc_top = sorted(results_kc, key=lambda r: r['sharpe'], reverse=True)[:3]
orb_top = sorted(results_orb, key=lambda r: r['sharpe'], reverse=True)[:3]

cross_results = []
for kc in kc_top:
    for orb in orb_top:
        config.STRATEGIES['keltner']['max_hold_bars'] = kc['kc_max_hold']
        r = run_variant(data, f"KC{kc['kc_max_hold']}_ORB{orb['orb_max_hold']}",
            **{**LIVE_KWARGS, "orb_max_hold_m15": orb['orb_max_hold']})
        r['kc_mh'] = kc['kc_max_hold']
        r['orb_mh'] = orb['orb_max_hold']
        cross_results.append(r)
        print(f"  KC{kc['kc_max_hold']}/ORB{orb['orb_max_hold']}: Sharpe={r['sharpe']:.2f}, PnL=${r['total_pnl']:.0f}")

config.STRATEGIES['keltner']['max_hold_bars'] = orig_mh
print_ranked(cross_results)

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp21_results.json', 'w') as f:
    json.dump(sanitize_for_json(results_kc + results_orb + cross_results), f, indent=2)
print('Saved')